# Реализация алгоритмов A2C и DDPG для решения задачи CarRacing-v3 в среде Gymnasium.

В работе реализуются алгоритмы Advantage Actor-Critic (A2C) и Deep Deterministic Policy Gradient (DDPG) для непрерывного управления автомобилем.
Оба агента обучаются на основе изображений трассы, после чего сравниваются результаты обучения и сохраняются модели.

### Установка зависимостей

Устанавливаем библиотеки для работы со средой CarRacing-v3, PyTorch и визуализацией. Если зависимости уже установлены, повторно выполнять эту ячейку не требуется.

In [ ]:
%pip install -q "gymnasium[box2d]>=1.1,<2" numpy matplotlib opencv-python torch tqdm imageio pillow nbformat

### Импорт библиотек

Подключаем библиотеки для работы со средой, нейронными сетями, вычислениями, графиками и GIF.


In [ ]:
import os, json, copy, random, math, shutil
from pathlib import Path
from collections import deque
import cv2
import numpy as np
import gymnasium as gym
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import imageio.v2 as imageio
from IPython.display import display, Image

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
torch.set_num_threads(min(4, torch.get_num_threads()))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
output_dir = Path('car_racing_results')
output_dir.mkdir(exist_ok=True)
print('Device:', device, 'PyTorch:', torch.__version__, 'Gymnasium:', gym.__version__)

### Создание среды

Создаём среду CarRacing-v3, выделяем признаки дорожного полотна из изображения и задаём плавное управление автомобилем.

In [ ]:
gamma_frame = 0.99
frame_skip = 3
rows = (17, 27, 37, 47, 57, 67)
bins = 8


def asphalt_mask(rgb):
    rgb = np.asarray(rgb, dtype=np.uint8)
    if rgb.shape != (96, 96, 3):
        raise ValueError(f'Expected CarRacing RGB (96, 96, 3), got {rgb.shape}')
    hi = rgb.max(axis=2).astype(np.int16)
    lo = rgb.min(axis=2).astype(np.int16)
    brightness = rgb.mean(axis=2)
    mask = ((hi - lo <= 24) & (brightness >= 55) & (brightness <= 225))
    return cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_CLOSE,
                            np.ones((3, 5), np.uint8))


def road_visibility(rgb):
    vec, _ = image_features(rgb)
    visible = [int(vec[i*(3+bins)+2] > 0.5) for i in range(len(rows))]
    return visible, float(asphalt_mask(rgb).mean())


def image_features(rgb, prev_gray=None, previous_action=(0., 0.6)):
    rgb = np.asarray(rgb, dtype=np.uint8)
    assert rgb.shape == (96, 96, 3), f'Expected (96,96,3), got {rgb.shape}'
    # Диагностика и агент используют одинаковую маску дорожного полотна.
    road = asphalt_mask(rgb)
    centers = []
    features = []
    reference_x = 48.0
    for y in reversed(rows):
        stripe = road[max(0, y - 1):min(96, y + 2)].mean(axis=0) >= 0.5
        padded = np.pad(stripe.astype(np.int8), (1, 1))
        edges = np.diff(padded)
        starts = np.flatnonzero(edges == 1)
        ends = np.flatnonzero(edges == -1)
        candidates = [(int(l), int(r)) for l, r in zip(starts, ends) if r-l >= 5]
        if candidates:
            left, right = min(candidates, key=lambda lr: abs((lr[0]+lr[1])/2-reference_x))
            center = (left+right)/2
            width = (right-left)/96.
            visible = 1.
            reference_x = center
        else:
            center = reference_x
            width = 0.
            visible = 0.
        centers.append(center)
        features.extend([(center-48.)/48., width, visible])

        features.extend([float(stripe[b*12:(b+1)*12].mean()) for b in range(bins)])
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    motion = (0.0 if prev_gray is None else float(np.abs(gray.astype(np.float32)-prev_gray.astype(np.float32)).mean()/128.))
    near_road = float(road[60:72, 43:53].mean())
    features.extend([float(road[12:75].mean()), near_road,
                     min(1., motion), float(previous_action[0]), float(previous_action[1])])
    result = np.asarray(features, dtype=np.float32)
    assert result.ndim == 1 and np.isfinite(result).all()
    return result, gray


nav_dim = 15
steer_step_limit = 0.38     # Максимальное изменение руля за один шаг управления.
gas_step_limit = 0.28
brake_step_limit = 0.45


def road_geometry(base_features):
    f = np.asarray(base_features, dtype=np.float32)
    assert f.ndim == 1 and len(f) == len(rows)*(3+bins)+5
    block = 3+bins
    centers = np.asarray([f[k*block] for k in range(len(rows))], dtype=np.float32)
    widths = np.asarray([f[k*block+1] for k in range(len(rows))], dtype=np.float32)
    visible = np.asarray([f[k*block+2] for k in range(len(rows))], dtype=np.float32)
    near, mid, far = float(centers[0]), float(centers[2]), float(centers[-1])
    heading = np.clip(far-near, -1., 1.)
    curvature = np.clip(far - 2.*mid + near, -1., 1.)
    geometry = np.array([near, heading, curvature], dtype=np.float32)
    confidence = float(np.mean(visible))
    return geometry, widths, visible, confidence, centers


def smooth_vehicle_action(requested, previous):
    target = np.clip(np.asarray(requested, dtype=np.float32),
                     [-1., 0., 0.], [1., 1., 1.]).astype(np.float32)
    prev = np.asarray(previous, dtype=np.float32)
    if target.shape != (3,) or prev.shape != (3,):
        raise ValueError('Steering, gas and brake must have shape (3,)')
    if not np.isfinite(target).all() or not np.isfinite(prev).all():
        raise ValueError('Non-finite action')
    limits = np.array([steer_step_limit, gas_step_limit, brake_step_limit],
                      dtype=np.float32)
    applied = prev + np.clip(target-prev, -limits, limits)
    applied = np.clip(applied, [-1., 0., 0.], [1., 1., 1.])
    # Не допускаем одновременного нажатия газа и тормоза после сглаживания.
    if target[2] > target[1]:
        applied[1] = 0.
    elif target[1] > target[2]:
        applied[2] = 0.
    else:
        applied[1:] = 0.
    return applied.astype(np.float32)


class RoadObservation(gym.Wrapper):
    def __init__(self, wrapped):
        super().__init__(wrapped)
        self.previous_gray = None
        self.previous_features = None
        self.last_applied = np.zeros(3, dtype=np.float32)
        self.last_geometry = np.zeros(3, dtype=np.float32)
        self.missed_road_steps = 10
        self.feature_size = len(image_features(np.full((96,96,3), 100, dtype=np.uint8))[0])
        self.observation_space = gym.spaces.Box(-2., 2.,
            shape=(self.feature_size*2+nav_dim,), dtype=np.float32)

    def _navigation(self, new_features, older_features):
        geometry, widths, visible, confidence, centers = road_geometry(new_features)
        if older_features is None:
            drift = np.zeros(3, dtype=np.float32)
        else:
            _, _, older_visible, _, older_centers = road_geometry(older_features)
            idx = (0, 2, 5)
            drift = np.asarray([
                float(np.clip(2.*(centers[k]-older_centers[k]),-1.,1.))
                if visible[k] > 0.5 and older_visible[k] > 0.5 else 0.
                for k in idx], dtype=np.float32)
        if visible.sum() >= 2 and visible[:4].sum() >= 1:
            self.last_geometry = geometry.copy()
            self.missed_road_steps = 0
        else:
            self.missed_road_steps = min(10, self.missed_road_steps+1)
        return np.concatenate([
            geometry,
            np.array([widths[0],confidence],dtype=np.float32),
            drift,
            self.last_geometry,
            np.array([self.missed_road_steps/10.],dtype=np.float32),
            self.last_applied
        ]).astype(np.float32)

    def reset(self, **kwargs):
        rgb, info = self.env.reset(**kwargs)
        self.last_applied = np.zeros(3, dtype=np.float32)
        self.last_geometry = np.zeros(3, dtype=np.float32)
        self.missed_road_steps = 10
        new_features, gray = image_features(rgb, None, (0., 0.))
        navigation = self._navigation(new_features, None)
        self.previous_features = new_features
        self.previous_gray = gray
        state = np.concatenate([new_features, new_features, navigation]).astype(np.float32)
        assert state.shape == self.observation_space.shape
        return state, info

    def step(self, action):
        requested = np.asarray(action, dtype=np.float32)
        applied = smooth_vehicle_action(requested, self.last_applied)
        raw_reward = 0.
        discounted_reward = 0.
        advanced = 0
        for k in range(frame_skip):
            rgb, reward, terminated, truncated, info = self.env.step(applied)
            raw_reward += float(reward)
            discounted_reward += (gamma_frame**k)*float(reward)
            advanced += 1
            if terminated or truncated:
                break
        self.last_applied = applied.copy()
        previous_features = self.previous_features
        new_features, gray = image_features(rgb, self.previous_gray,
                            (float(applied[0]),float(applied[1]-applied[2])))
        navigation = self._navigation(new_features, previous_features)
        state = np.concatenate([new_features, previous_features, navigation]).astype(np.float32)
        assert state.shape == self.observation_space.shape and np.isfinite(state).all()
        self.previous_features = new_features
        self.previous_gray = gray
        info = dict(info)
        info.update(discounted_reward=float(discounted_reward),
                    decision_discount=float(gamma_frame**advanced),
                    frames_advanced=advanced,
                    requested_action=requested.tolist(),
                    applied_action=applied.tolist(),
                    road_visibility=float(navigation[4]),
                    road_age=float(navigation[11]))
        return state, float(raw_reward), terminated, truncated, info


def make_env(render_mode=None):
    return RoadObservation(gym.make('CarRacing-v3', continuous=True, render_mode=render_mode))

obs_dim = 2*len(image_features(np.full((96,96,3), 100, dtype=np.uint8))[0])+nav_dim

### Проверка распознавания дороги

Проверяем, что дорожное полотно определяется на кадрах после начала движения автомобиля.


In [ ]:
# Проверяем распознавание дороги после запуска симуляции.
diagnostic_env = gym.make('CarRacing-v3', continuous=True)
try:
    diagnostic_rgb, _ = diagnostic_env.reset(seed=seed)
    snapshots = []
    for frame in range(1, 65):
        diagnostic_rgb, _, terminated, truncated, _ = diagnostic_env.step(
            np.array([0., 0.55, 0.], dtype=np.float32)
        )
        if frame in (12, 24, 36, 48, 64):
            snapshots.append((frame, np.asarray(diagnostic_rgb).copy()))
        if terminated or truncated:
            break

    checked = [(frame, rgb, *road_visibility(rgb)) for frame, rgb in snapshots]
    valid = [item for item in checked if sum(item[2]) >= 2]
    if not valid:
        raise RuntimeError(
            'Дорога не обнаружена после начала движения. '
            'Проверьте маску и параметры среды до запуска обучения.'
        )

    selected = sorted(valid, key=lambda item: (sum(item[2]), item[3]),
                      reverse=True)[:2]
    fig, axs = plt.subplots(len(selected), 2,
                            figsize=(9, 3.6 * len(selected)), squeeze=False)
    for index, (frame, rgb, visible, coverage) in enumerate(selected):
        axs[index, 0].imshow(rgb)
        axs[index, 0].set_title(f'RGB · кадр {frame}')
        axs[index, 1].imshow(asphalt_mask(rgb), cmap='gray', vmin=0, vmax=1)
        axs[index, 1].set_title(f'Маска дороги · кадр {frame}')
        for ax in axs[index]:
            for y in rows:
                ax.axhline(y, alpha=.23, color='cyan')
            ax.axis('off')
    plt.tight_layout()
    plt.show()
    print('Проверка распознавания дороги пройдена успешно.')
finally:
    diagnostic_env.close()


### Параметры обучения

Задаём основные параметры среды и алгоритмов A2C и DDPG.


In [ ]:
episode_n = 200
t_max = 334
rollout = 64
gae_lambda = 0.95
reward_scale = 0.1          # Масштабируем награду только для обучения; на графиках — исходная.
value_coef = 0.5
entropy_coef_start = 0.01
entropy_coef_end = 0.003
a2c_actor_lr = 1.0e-4
a2c_value_lr = 3.0e-4
actor_lr = 1e-4
critic_lr = 2e-4
batch_size = 128
warmup_steps = 1500
replay_capacity = 30_000
ddpg_noise_steer = 0.22
ddpg_noise_long = 0.12
tau = 0.005
eval_every = 10
kl_limit = 0.025
ddpg_update_every = 2
ddpg_policy_delay = 2
validation_seeds = (12001, 12002, 12003)
test_seeds = tuple(range(22001, 22011))
window = 20
assert episode_n == 200 and t_max * frame_skip >= 1000


def obs_tensor(observation):
    return torch.as_tensor(np.asarray(observation), dtype=torch.float32, device=device)


def raw_to_action(raw):
    steering, longitudinal = raw[..., 0], raw[..., 1]
    return torch.stack([steering, torch.relu(longitudinal), 0.5*torch.relu(-longitudinal)], dim=-1)


def numpy_action(raw):
    steer, long = np.asarray(raw, dtype=np.float32)
    return np.array([steer, max(0., long), 0.5*max(0., -long)], dtype=np.float32)


def initial_bias():

    return torch.tensor([0., float(np.arctanh(0.65))], dtype=torch.float32)


def episode_seed(episode):
    if episode < 40:
        return seed + (episode % 8)
    if episode < 100:
        return seed + (episode % 24)
    if episode % 4 == 0:
        return seed + 200 + episode
    return seed + (episode % 32)


def finite_gradients(module):
    return all(torch.isfinite(p.grad).all().item() for p in module.parameters() if p.grad is not None)

### Архитектура A2C

Создаём Actor для выбора непрерывных действий и Value Network для оценки состояния среды.


In [ ]:
class MLP(nn.Module):
    def __init__(self, input_size=obs_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(input_size, 256), nn.LayerNorm(256), nn.ReLU(),
                                 nn.Linear(256, hidden), nn.ReLU())
    def forward(self, x):
        return self.net(x)


class ActorNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = MLP()
        self.head = nn.Linear(128, 2)
        self.log_std = nn.Parameter(torch.tensor([-1.0, -1.3]))
        with torch.no_grad():
            self.head.bias.copy_(initial_bias())
    def forward(self, s):
        return self.head(self.features(s)), self.log_std.clamp(-3., -0.45).exp()
    def sample(self, s, deterministic=False):
        mu, std = self(s)
        dist = Normal(mu, std)
        # sample() обеспечивает корректный градиент стохастической политики.
        z = mu if deterministic else dist.sample()
        raw = torch.tanh(z)

        logp = (dist.log_prob(z)-torch.log(1-raw.square()+1e-6)).sum(dim=-1)
        entropy_proxy = dist.entropy().sum(dim=-1)
        return raw_to_action(raw), logp, entropy_proxy, raw


class ValueNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = MLP()
        self.head = nn.Linear(128, 1)
    def forward(self, s):
        return self.head(self.features(s)).squeeze(-1)


class A2CAgent(nn.Module):
    def __init__(self):
        super().__init__()
        self.actor = ActorNet()
        self.critic = ValueNet()
    def action(self, state, deterministic=False):
        return self.actor.sample(state, deterministic)[0]

### Архитектура DDPG и Replay Buffer

Создаём Actor, Critic, целевые сети и Replay Buffer для хранения переходов агента.


In [ ]:
class DDPGActor(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = MLP()
        self.head = nn.Linear(128, 2)
        with torch.no_grad():
            self.head.bias.copy_(initial_bias())
    def forward(self, state):
        return raw_to_action(torch.tanh(self.head(self.features(state))))
    def raw(self, state):
        return torch.tanh(self.head(self.features(state)))


class DDPGCritic(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(obs_dim+3, 256), nn.LayerNorm(256), nn.ReLU(),
                                 nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 1))
    def forward(self, state, action):
        return self.net(torch.cat([state, action], dim=-1)).squeeze(-1)


class ReplayBuffer:
    def __init__(self, capacity=replay_capacity):
        self.data = deque(maxlen=capacity)
    def append(self, state, action, scaled_reward, next_state, terminal, discount):
        self.data.append((np.array(state, copy=True), np.array(action, copy=True),
                          float(scaled_reward), np.array(next_state, copy=True),
                          float(terminal), float(discount)))
    def sample(self, batch_size=batch_size):
        sampled = random.sample(self.data, batch_size)
        s, a, r, ns, terminals, discounts = zip(*sampled)
        return (obs_tensor(np.stack(s)), obs_tensor(np.stack(a)),
                obs_tensor(r), obs_tensor(np.stack(ns)),
                obs_tensor(terminals), obs_tensor(discounts))
    def __len__(self):
        return len(self.data)


def soft_update(target, model, tau=tau):
    with torch.no_grad():
        for target_parameter, online_parameter in zip(target.parameters(), model.parameters()):
            target_parameter.mul_(1-tau).add_(online_parameter, alpha=tau)


class ExplorationNoise:
    def __init__(self):
        self.value = np.zeros(2, dtype=np.float32)
    def reset(self):
        self.value[:] = 0.
    def sample(self, scale):
        # Коррелированный шум для исследования непрерывных действий.
        self.value = 0.75*self.value + 0.40*np.random.randn(2).astype(np.float32)
        return self.value*scale


def ddpg_exploration(state, actor, global_step, noise):

    if global_step < warmup_steps:
        raw = np.array([np.random.normal(0., 0.22), np.random.uniform(0.40, 0.80)], dtype=np.float32)
    else:
        with torch.no_grad():
            raw = actor.raw(obs_tensor(state[None]))[0].cpu().numpy()
        progress = max(0., min(1., (global_step-warmup_steps)/(episode_n*t_max)))
        amplitude = 1.0-0.65*progress
        raw = raw + noise.sample(np.array([ddpg_noise_steer, ddpg_noise_long], dtype=np.float32))*amplitude
    return numpy_action(np.clip(raw, -1., 1.))

### Проверка алгоритмов

Проверяем размерности данных, вычисление градиентов и корректность взаимодействия со средой перед обучением.

In [ ]:
def gae_targets(rewards, values, next_values, terminated, done, discounts, lam=gae_lambda):

    with torch.no_grad():
        values = values.detach()
        next_values = next_values.detach()
        advantages = torch.zeros_like(values)
        gae = torch.zeros((), device=values.device)
        for t in reversed(range(len(rewards))):
            delta = rewards[t] + discounts[t]*(1.-terminated[t])*next_values[t]-values[t]
            gae = delta+discounts[t]*lam*(1.-done[t])*gae
            advantages[t] = gae
        returns = advantages + values
    return advantages, returns


def self_tests():
    rgb = np.zeros((96,96,3),dtype=np.uint8)
    rgb[:,:] = [105,190,105]
    rgb[10:75,34:64] = [105,105,105]
    feats, gray = image_features(rgb)
    assert feats.shape == ((obs_dim-nav_dim)//2,) and np.isfinite(feats).all()
    assert feats[-4] > 0.5, 'Road should be detected at car center'


    test_mask = asphalt_mask(rgb)
    assert test_mask[40,48] == 1 and test_mask[40,15] == 0
    marked, coverage = road_visibility(rgb)
    assert sum(marked) >= 2 and coverage > 0.1

    nav_geometry, _, _, conf, _ = road_geometry(feats)
    assert conf >= 0.5 and np.isfinite(nav_geometry).all()


    current_action = np.zeros(3,dtype=np.float32)
    commanded = np.array([1.,1.,0.],dtype=np.float32)
    applied = smooth_vehicle_action(commanded,current_action)
    assert 0. < applied[0] <= steer_step_limit+1e-6
    assert 0. < applied[1] <= gas_step_limit+1e-6
    braked = smooth_vehicle_action(np.array([-1.,0.,1.],dtype=np.float32),applied)
    assert abs(braked[0]-applied[0]) <= steer_step_limit+1e-6
    assert braked[1] == 0. and braked[2] > 0.
    fake_nav = np.r_[nav_geometry,0.4,conf,np.zeros(3),nav_geometry,0.0,applied]
    assert fake_nav.shape == (nav_dim,)
    x = torch.from_numpy(np.stack([np.r_[feats,feats,fake_nav]]*4).astype(np.float32)).to(device)
    agent = A2CAgent().to(device)
    action, logp, entropy, raw = agent.actor.sample(x)
    assert action.shape == (4,3) and logp.shape == (4,) and torch.isfinite(logp).all()
    assert (action[:,1]*action[:,2] == 0).all(), 'Gas and brake cannot overlap'
    agent.actor.zero_grad(set_to_none=True)
    (-logp.mean()).backward()
    assert agent.actor.head.weight.grad.abs().sum() > 0
    agent.critic.zero_grad(set_to_none=True)
    agent.critic(x).square().mean().backward()
    assert agent.critic.head.weight.grad.abs().sum() > 0
    ac, q = DDPGActor().to(device), DDPGCritic().to(device)
    ac.zero_grad(set_to_none=True)
    loss = -q(x,ac(x)).mean()
    loss.backward()
    assert ac.head.weight.grad.abs().sum() > 0
    rb = ReplayBuffer(10)
    for i in range(5):
        rb.append(x[0].cpu().numpy(), action[0].detach().cpu().numpy(), -0.1, x[1].cpu().numpy(), i==0, gamma_frame**3)
    states, actions, rews, next_states, terms, discounts = rb.sample(4)
    assert states.shape == (4,obs_dim) and actions.shape == (4,3)
    vals = torch.zeros(2,device=device)
    av, ret = gae_targets(torch.tensor([1.,2.],device=device), vals, vals,
                          torch.tensor([0.,1.],device=device),
                          torch.tensor([0.,1.],device=device),
                          torch.ones(2,device=device), lam=1.)
    assert torch.allclose(ret, torch.tensor([3.,2.],device=device))
    # Проверяем формы тензоров, чтобы исключить ошибку broadcasting.
    test_returns = torch.tensor([1., 2., 3.], device=device)
    test_values = torch.zeros(3, device=device)
    test_advantage = test_returns - test_values
    assert test_advantage.shape == (3,)
    assert agent.critic(x).shape == (4,)
    # При завершении эпизода bootstrap отключается; при ограничении времени сохраняется.
    r = torch.tensor([1.], device=device)
    v = torch.tensor([0.], device=device)
    nv = torch.tensor([7.], device=device)
    d = torch.tensor([0.9], device=device)
    term, _ = gae_targets(r, v, nv, torch.ones_like(v), torch.ones_like(v), d)
    trunc, _ = gae_targets(r, v, nv, torch.zeros_like(v), torch.ones_like(v), d)
    assert torch.allclose(term, r) and torch.allclose(trunc, r + 0.9*nv)

    sample_logp = torch.randn(4,device=device,requires_grad=True)
    sample_adv = torch.randn(4,device=device)
    test_policy_loss = -(sample_logp * sample_adv.detach()).mean()
    assert test_policy_loss.ndim == 0
    test_policy_loss.backward()
    assert torch.isfinite(sample_logp.grad).all()

self_tests()


_test_env = make_env()
try:
    state, _ = _test_env.reset(seed=seed)
    assert state.shape == (obs_dim,) and np.isfinite(state).all()
    last_applied_steer = 0.0
    for k in range(4):
        command = np.array([1. if k%2==0 else -1.,0.65,0.],dtype=np.float32)
        state, reward, terminated, truncated, info = _test_env.step(command)
        performed = np.asarray(info['applied_action'],dtype=np.float32)
        assert abs(performed[0] - last_applied_steer) <= steer_step_limit+1e-6
        assert np.allclose(state[-3:],performed)
        last_applied_steer = performed[0]
        assert np.isfinite(reward) and state.shape == (obs_dim,)
        assert 1 <= info['frames_advanced'] <= frame_skip
        if terminated or truncated: break
finally:
    _test_env.close()

### Оценка агентов

Проверяем качество политики на отдельных трассах без исследовательского шума и сохраняем лучшие веса нейросетей.

In [ ]:
def evaluate_agent(agent, algorithm, seeds=validation_seeds):
    test_env = make_env()
    previous_mode = agent.training
    agent.eval()
    scores, mean_actions = [], []
    try:
        with torch.no_grad():
            for seed in seeds:
                state, _ = test_env.reset(seed=int(seed))
                score, actions = 0., []
                for decision in range(t_max):
                    x = obs_tensor(state[None])
                    action = (agent.action(x, deterministic=True)[0] if algorithm=='a2c'
                              else agent(x)[0]).cpu().numpy().astype(np.float32)
                    actions.append(action)
                    state, reward, terminated, truncated, _ = test_env.step(action)
                    score += float(reward)
                    if terminated or truncated: break
                scores.append(score)
                mean_actions.append(np.mean(actions,axis=0))
    finally:
        test_env.close()
        agent.train(previous_mode)
    return float(np.mean(scores)), [float(x) for x in scores], np.mean(mean_actions,axis=0).tolist()


def write_json(path, data):
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def save_best(path, algorithm, agent, episode, score):
    torch.save({'algorithm': algorithm, 'episode': episode, 'validation_mean': score,
                'model': copy.deepcopy(agent.state_dict()), 'observation_dim': obs_dim}, path)

### Обучение A2C

Запускаем обучение агента A2C на 200 эпизодах и сохраняем награды и результаты промежуточных проверок.

In [ ]:
def train_a2c(episode_count=episode_n):
    agent = A2CAgent().to(device)
    actor_optimizer = torch.optim.Adam(agent.actor.parameters(), lr=a2c_actor_lr, eps=1e-5)
    value_optimizer = torch.optim.Adam(agent.critic.parameters(), lr=a2c_value_lr, eps=1e-5)
    environment = make_env()
    rewards, validation = [], []
    best_score = -float('inf')
    actor_lr_factor = 1.0
    discarded_updates = 0
    progress_bar = tqdm(range(episode_count), desc='A2C')
    try:
        for episode in progress_bar:
            state, _ = environment.reset(seed=episode_seed(episode))
            episode_reward = 0.
            decisions = 0
            completed = False
            while not completed and decisions < t_max:
                logps, entropies, values = [], [], []
                old_means, old_stds, policy_states = [], [], []
                scaled_rewards, discounts, terminals, ends, future_values = [], [], [], [], []
                for _ in range(rollout):
                    current = obs_tensor(state[None])
                    action, logp, entropy, _ = agent.actor.sample(current)
                    value = agent.critic(current)
                    with torch.no_grad():
                        old_mean, old_std = agent.actor(current)
                    old_means.append(old_mean.detach())
                    old_stds.append(old_std.detach().expand_as(old_mean))
                    policy_states.append(current.detach())
                    next_state, reward, terminated, truncated, info = environment.step(action[0].detach().cpu().numpy())
                    decisions += 1
                    done = bool(terminated or truncated or decisions >= t_max)
                    with torch.no_grad():
                        next_value = (torch.zeros((), device=device) if terminated
                                      else agent.critic(obs_tensor(next_state[None]))[0])
                    logps.append(logp[0]); entropies.append(entropy[0]); values.append(value[0])
                    scaled_rewards.append(float(info['discounted_reward'])*reward_scale)
                    discounts.append(float(info['decision_discount']))
                    terminals.append(float(terminated))
                    ends.append(float(done))
                    future_values.append(next_value)
                    state = next_state
                    episode_reward += float(reward)
                    completed = done
                    if completed: break

                value_tensor = torch.stack(values)
                advantage, value_targets = gae_targets(
                    obs_tensor(scaled_rewards), value_tensor, torch.stack(future_values),
                    obs_tensor(terminals), obs_tensor(ends), obs_tensor(discounts))
                assert value_tensor.ndim == advantage.ndim == value_targets.ndim == 1
                assert len(value_tensor) == len(logps) == len(advantage)

                policy_weights = advantage/(1.0+advantage.square().mean().sqrt())
                entropy_weight = entropy_coef_end + (entropy_coef_start-entropy_coef_end)*(1.-episode/max(1,episode_count-1))
                actor_loss = -(torch.stack(logps)*policy_weights.detach()).mean() - entropy_weight*torch.stack(entropies).mean()
                value_loss = F.smooth_l1_loss(value_tensor, value_targets.detach())

                actor_snapshot = copy.deepcopy(agent.actor.state_dict())
                actor_optimizer.zero_grad(set_to_none=True)
                actor_loss.backward()
                nn.utils.clip_grad_norm_(agent.actor.parameters(), 0.5)
                if not torch.isfinite(actor_loss) or not finite_gradients(agent.actor):
                    raise FloatingPointError(f'Non-finite A2C actor gradients at episode {episode+1}')
                actor_optimizer.step()
                # Отменяем слишком большие обновления Actor по оценке KL.

                with torch.no_grad():
                    states_batch = torch.cat(policy_states, dim=0)
                    mu_new, std_new = agent.actor(states_batch)
                    mu_old = torch.cat(old_means, dim=0)
                    std_old = torch.cat(old_stds, dim=0)
                    kl = torch.distributions.kl_divergence(
                        Normal(mu_old, std_old), Normal(mu_new, std_new.expand_as(mu_new))
                    ).sum(-1).mean()
                if not torch.isfinite(kl).item() or kl.item() > kl_limit:
                    agent.actor.load_state_dict(actor_snapshot)
                    actor_optimizer.state.clear()
                    actor_lr_factor = max(0.25, actor_lr_factor * 0.85)
                    discarded_updates += 1

                value_optimizer.zero_grad(set_to_none=True)
                (value_coef*value_loss).backward()
                nn.utils.clip_grad_norm_(agent.critic.parameters(), 1.0)
                if not torch.isfinite(value_loss) or not finite_gradients(agent.critic):
                    raise FloatingPointError(f'Non-finite A2C value gradients at episode {episode+1}')
                value_optimizer.step()


            lr_scale = 1.0 - 0.55*(episode/max(1, episode_count-1))
            for group in actor_optimizer.param_groups: group['lr'] = a2c_actor_lr*lr_scale*actor_lr_factor
            for group in value_optimizer.param_groups: group['lr'] = a2c_value_lr*lr_scale
            rewards.append(episode_reward)
            moving_mean = float(np.mean(rewards[-window:]))
            if (episode+1) % eval_every == 0 or episode+1 == episode_count:
                score, individual, actions = evaluate_agent(agent,'a2c')
                validation.append({'episode': episode+1,'mean':score,'scores':individual,'mean_action':actions})
                print(f'A2C {episode+1:3d}/{episode_count} | средняя награда (20): {moving_mean:+.1f} | валидация: {score:+.1f}')
                if score > best_score:
                    best_score = score
                    save_best(output_dir/'a2c_best.pth','a2c',agent,episode+1,score)
                # Лучшую модель сохраняем отдельно, не откатывая обучаемую политику.


                write_json(output_dir/'a2c_validation.json',validation)
            if (episode+1) % 20 == 0 or episode+1 == episode_count:
                torch.save({'algorithm':'a2c','episode':episode+1,'model':agent.state_dict()},output_dir/'a2c_last.pth')
                np.save(output_dir/'a2c_rewards.npy', np.asarray(rewards,dtype=np.float32))
    finally:
        environment.close()
    assert len(rewards)==episode_count
    return agent, rewards, validation

a2c_model, a2c_rewards, a2c_validation = train_a2c()

### Обучение DDPG

Обучаем агента DDPG на 200 эпизодах с использованием Replay Buffer и целевых сетей.


In [ ]:
def train_ddpg(episode_count=episode_n):
    actor, critic = DDPGActor().to(device), DDPGCritic().to(device)
    target_actor, target_critic = copy.deepcopy(actor), copy.deepcopy(critic)
    actor_optim = torch.optim.Adam(actor.parameters(),lr=actor_lr)
    critic_optim = torch.optim.Adam(critic.parameters(),lr=critic_lr)
    buffer = ReplayBuffer()
    env_train = make_env()
    noise = ExplorationNoise()
    rewards, validation = [], []
    global_step, best_score = 0, -float('inf')
    critic_updates = 0
    progress_bar = tqdm(range(episode_count), desc='DDPG')
    try:
        for episode in progress_bar:
            state, _ = env_train.reset(seed=episode_seed(episode))
            noise.reset()
            ep_reward, decisions = 0., 0
            done = False
            while not done and decisions < t_max:
                action = ddpg_exploration(state,actor,global_step,noise)
                new_state, reward, terminated, truncated, info = env_train.step(action)
                decisions += 1
                global_step += 1
                done = bool(terminated or truncated or decisions>=t_max)
                buffer.append(state,action,float(info['discounted_reward'])*reward_scale,
                              new_state,float(terminated),float(info['decision_discount']))
                state = new_state
                ep_reward += float(reward)
                if (global_step>=warmup_steps and len(buffer)>=batch_size
                        and global_step % ddpg_update_every == 0):
                    s,a,r,ns,term,disc = buffer.sample(batch_size)
                    with torch.no_grad():
                        target_q = r+disc*(1.-term)*target_critic(ns,target_actor(ns))
                    q_pred = critic(s,a)
                    assert q_pred.shape == target_q.shape == r.shape
                    critic_loss = F.smooth_l1_loss(q_pred,target_q)
                    critic_optim.zero_grad(set_to_none=True)
                    critic_loss.backward()
                    nn.utils.clip_grad_norm_(critic.parameters(),1.0)
                    if not torch.isfinite(critic_loss) or not finite_gradients(critic):
                        raise FloatingPointError(f'DDPG critic NaN on episode {episode+1}')
                    critic_optim.step()
                    critic_updates += 1
                    if critic_updates % ddpg_policy_delay != 0:
                        continue
                    # При обучении Actor сохраняем градиент по действию, но не обновляем Critic.
                    for p in critic.parameters(): p.requires_grad_(False)
                    actor_loss = -critic(s,actor(s)).mean()
                    actor_optim.zero_grad(set_to_none=True)
                    actor_loss.backward()
                    nn.utils.clip_grad_norm_(actor.parameters(),1.0)
                    if not torch.isfinite(actor_loss) or not finite_gradients(actor):
                        raise FloatingPointError(f'DDPG actor NaN on episode {episode+1}')
                    actor_optim.step()
                    for p in critic.parameters(): p.requires_grad_(True)
                    soft_update(target_actor,actor)
                    soft_update(target_critic,critic)

            rewards.append(ep_reward)
            mean20 = float(np.mean(rewards[-window:]))
            if (episode+1)%eval_every==0 or episode+1==episode_count:
                score, indiv, actions = evaluate_agent(actor,'ddpg')
                validation.append({'episode':episode+1,'mean':score,'scores':indiv,'mean_action':actions})
                print(f'DDPG {episode+1:3d}/{episode_count} | средняя награда (20): {mean20:+.1f} | валидация: {score:+.1f}')
                if score > best_score:
                    best_score = score
                    # Лучшую модель сохраняем отдельно, без отката рабочих сетей.
                    torch.save({'algorithm':'ddpg','episode':episode+1,
                                'validation_mean':score,'model':copy.deepcopy(actor.state_dict()),
                                'critic':copy.deepcopy(critic.state_dict()),
                                'target_actor':copy.deepcopy(target_actor.state_dict()),
                                'target_critic':copy.deepcopy(target_critic.state_dict())},
                               output_dir/'ddpg_best.pth')


                write_json(output_dir/'ddpg_validation.json', validation)
            for group in actor_optim.param_groups:
                group['lr'] = actor_lr * (1 - .50*episode/max(1,episode_count-1))
            for group in critic_optim.param_groups:
                group['lr'] = critic_lr * (1 - .35*episode/max(1,episode_count-1))
            if (episode+1)%20==0 or episode+1==episode_count:
                torch.save({'algorithm':'ddpg','episode':episode+1,'model':actor.state_dict()},output_dir/'ddpg_last.pth')

                torch.save({'episode':episode+1,'actor':actor.state_dict(),'critic':critic.state_dict(),
                            'target_actor':target_actor.state_dict(),'target_critic':target_critic.state_dict(),
                            'actor_optim':actor_optim.state_dict(),'critic_optim':critic_optim.state_dict()},
                           output_dir/'ddpg_network_checkpoint.pth')
                np.save(output_dir/'ddpg_rewards.npy',np.asarray(rewards,dtype=np.float32))
    finally:
        env_train.close()
    assert len(rewards)==episode_count
    return actor, rewards, validation

ddpg_actor, ddpg_rewards, ddpg_validation = train_ddpg()

### Результаты обучения

Строим отдельные графики награды A2C и DDPG, скользящей средней и независимой оценки, а также сравниваем результаты алгоритмов.


In [ ]:
def moving_average(rewards, window=window):
    rewards = np.asarray(rewards, dtype=np.float64)
    assert rewards.ndim == 1 and len(rewards) > 0
    return np.asarray([rewards[max(0, i-window+1):i+1].mean()
                       for i in range(len(rewards))])


def plot_one_model(name, rewards, validation):
    x = np.arange(1, len(rewards)+1)
    fig, ax = plt.subplots(figsize=(11, 4.5))
    ax.plot(x, rewards, alpha=0.25, label='Награда эпизода', linewidth=1)
    ax.plot(x, moving_average(rewards), linewidth=2.2,
            label=f'Скользящая средняя ({window} эп.)')
    ax.set(xlabel='Эпизод', ylabel='Награда CarRacing-v3',
           title=f'{name}: обучение ({len(rewards)} эпизодов)')
    ax.legend(loc='best'); ax.grid(alpha=0.25)
    fig.tight_layout()
    plot_file = output_dir / f'{name.lower()}_training.png'
    fig.savefig(plot_file, dpi=170)
    plt.show()
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(11, 4.5))
    ax.plot([item['episode'] for item in validation],
            [item['mean'] for item in validation], marker='o', linewidth=1.8,
            label='Валидация на фиксированных трассах')
    best_item = max(validation, key=lambda item: item['mean'])
    ax.axvline(best_item['episode'], linestyle=':', alpha=0.7,
               label=f"Лучшие веса: эпизод {best_item['episode']}")
    ax.set(xlabel='Эпизод', ylabel='Средняя реальная награда',
           title=f'{name}: независимая валидация')
    ax.legend(loc='best'); ax.grid(alpha=0.25)
    fig.tight_layout()
    validation_file = output_dir / f'{name.lower()}_validation.png'
    fig.savefig(validation_file, dpi=170)
    plt.show()
    plt.close(fig)
    return plot_file, validation_file


a2c_plot_files = plot_one_model('A2C', a2c_rewards, a2c_validation)
ddpg_plot_files = plot_one_model('DDPG', ddpg_rewards, ddpg_validation)


fig, ax = plt.subplots(figsize=(11, 4.5))
for name, rewards in [('A2C', a2c_rewards), ('DDPG', ddpg_rewards)]:
    ax.plot(np.arange(1, len(rewards)+1), moving_average(rewards),
            linewidth=2, label=f'{name}: mean{window}')
ax.set(xlabel='Эпизод', ylabel='Скользящая средняя награда',
       title='CarRacing-v3: сравнение A2C и DDPG')
ax.grid(alpha=.25); ax.legend()
fig.tight_layout()
fig.savefig(output_dir/'learning_curves.png', dpi=170)
plt.show(); plt.close(fig)

fig, ax = plt.subplots(figsize=(11, 4.5))
for name, history in [('A2C', a2c_validation), ('DDPG', ddpg_validation)]:
    ax.plot([point['episode'] for point in history],
            [point['mean'] for point in history], marker='o', label=name)
ax.set(xlabel='Эпизод', ylabel='Средняя реальная награда',
       title='Независимая валидация A2C и DDPG')
ax.grid(alpha=.25); ax.legend()
fig.tight_layout()
fig.savefig(output_dir/'validation_curves.png', dpi=170)
plt.show(); plt.close(fig)
print('Отдельные графики сохранены в:', output_dir)


### Тестирование агентов

Загружаем лучшие сохранённые модели и проверяем их на трассах, не использовавшихся для обучения и выбора весов.


In [ ]:
a2c_saved = torch.load(output_dir/'a2c_best.pth', map_location=device, weights_only=True)
ddpg_saved = torch.load(output_dir/'ddpg_best.pth', map_location=device, weights_only=True)
a2c_model.load_state_dict(a2c_saved['model'])
ddpg_actor.load_state_dict(ddpg_saved['model'])
print('A2C: лучший эпизод:', a2c_saved['episode'],
      'валидация:', round(a2c_saved['validation_mean'],2))
print('DDPG: лучший эпизод:', ddpg_saved['episode'],
      'валидация:', round(ddpg_saved['validation_mean'],2))

a2c_test_mean, a2c_test_scores, _ = evaluate_agent(a2c_model, 'a2c', test_seeds)
ddpg_test_mean, ddpg_test_scores, _ = evaluate_agent(ddpg_actor, 'ddpg', test_seeds)
print(f'A2C:  тестовая награда {a2c_test_mean:.2f} ± {np.std(a2c_test_scores):.2f}')
print(f'DDPG: тестовая награда {ddpg_test_mean:.2f} ± {np.std(ddpg_test_scores):.2f}')
write_json(output_dir/'a2c_test.json', dict(seeds=list(test_seeds), scores=a2c_test_scores,
                                           mean=a2c_test_mean))
write_json(output_dir/'ddpg_test.json', dict(seeds=list(test_seeds), scores=ddpg_test_scores,
                                            mean=ddpg_test_mean))


### Визуализация

Записываем заезды обученных агентов A2C и DDPG и сохраняем две GIF-анимации.


In [ ]:
def render_gif(agent, algorithm, seed=22030, frame_stride=2, display_scale=2):
    env_demo = make_env(render_mode='rgb_array')
    frames = []
    previous_mode = agent.training
    agent.eval()
    score = 0.0

    def capture():
        rgb = env_demo.render()
        if rgb is None:
            raise RuntimeError("render() вернул None: проверьте render_mode='rgb_array'")
        rgb = np.asarray(rgb, dtype=np.uint8)
        if rgb.ndim != 3 or rgb.shape[-1] != 3:
            raise ValueError(f'Ожидался RGB кадр, получен shape={rgb.shape}')
        if display_scale != 1:
            rgb = cv2.resize(rgb, (rgb.shape[1]*display_scale,
                                   rgb.shape[0]*display_scale),
                             interpolation=cv2.INTER_NEAREST)
        frames.append(np.ascontiguousarray(rgb))

    try:
        state, _ = env_demo.reset(seed=seed)
        capture()
        for t in range(t_max):
            with torch.no_grad():
                obs = obs_tensor(state[None])
                if algorithm == 'a2c':
                    action = agent.action(obs, deterministic=True)[0].cpu().numpy()
                elif algorithm == 'ddpg':
                    action = agent(obs)[0].cpu().numpy()
                else:
                    raise ValueError(f'Unknown algorithm: {algorithm}')
            state, reward, terminated, truncated, _ = env_demo.step(action)
            score += float(reward)
            if t % frame_stride == 0 or terminated or truncated or t == t_max-1:
                capture()
            if terminated or truncated:
                break
    finally:
        env_demo.close()
        agent.train(previous_mode)

    path = output_dir/f'{algorithm}_drive.gif'
    imageio.mimsave(path, frames, fps=14, loop=0)
    assert path.exists() and path.stat().st_size > 0
    return path, score


gif_scores = {}
for name, model in [('a2c', a2c_model), ('ddpg', ddpg_actor)]:
    gif_file, gif_reward = render_gif(model, name)
    gif_scores[name] = gif_reward
    print(f'{name.upper()}: GIF сохранён в {gif_file}, награда на этой трассе = {gif_reward:.1f}')
    display(Image(filename=str(gif_file)))


### Сохранение агентов

Сохраняем веса обученных моделей, графики, GIF, историю наград и результаты тестирования для дальнейшего использования.

In [ ]:
import csv


def export_rewards_csv(name, rewards):
    path = output_dir/f'{name.lower()}_rewards.csv'
    ma = moving_average(rewards)
    with path.open('w', encoding='utf-8', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['episode', 'reward', f'mean_{window}'])
        for i, (reward, average) in enumerate(zip(rewards, ma), start=1):
            writer.writerow([i, float(reward), float(average)])
    return path


def report(name, rewards, validation, test):
    first = float(np.mean(rewards[:window]))
    last = float(np.mean(rewards[-window:]))
    best = max(validation, key=lambda x: x['mean'])
    return {'algorithm': name, 'episodes': len(rewards), 'first_20': first,
            'last_20': last, 'delta_20': last-first,
            'best_validation': float(best['mean']),
            'best_episode': int(best['episode']), 'test_mean': float(test),
            'gif_score': float(gif_scores[name.lower()])}

export_rewards_csv('A2C', a2c_rewards)
export_rewards_csv('DDPG', ddpg_rewards)
summary = [report('A2C', a2c_rewards, a2c_validation, a2c_test_mean),
           report('DDPG', ddpg_rewards, ddpg_validation, ddpg_test_mean)]
write_json(output_dir/'summary.json', summary)
write_json(output_dir/'config.json', {
    'environment': 'CarRacing-v3', 'episodes_per_algorithm': episode_n,
    'observation': 'RGB-derived road features', 'observation_dim': obs_dim,
    'frame_skip': frame_skip, 'gamma_frame': gamma_frame,
    'validation_seeds': list(validation_seeds),
    'test_seeds': list(test_seeds), 'gif_seed': 22030})

for item in summary:
    print(f"{item['algorithm']}: первые 20 = {item['first_20']:.1f}, "
          f"последние 20 = {item['last_20']:.1f}, "
          f"изменение = {item['delta_20']:+.1f}, "
          f"лучшая валидация = {item['best_validation']:.1f} "
          f"(эп. {item['best_episode']}), тест = {item['test_mean']:.1f}")
    if item['delta_20'] <= 0:
        print(f"ВНИМАНИЕ: средняя награда {item['algorithm']} не выросла — "
              'рост средней награды в этом запуске не подтверждён.')

zip_path = shutil.make_archive(str(output_dir), 'zip', root_dir=str(output_dir))
print('Архив результатов:', zip_path)

## Вывод

Реализованы и обучены алгоритмы A2C и DDPG для непрерывного управления автомобилем в среде CarRacing-v3.

Для каждого алгоритма проведено 200 эпизодов обучения. Изменение награды показано на графиках, работа лучших сохранённых моделей проверяется на отдельных трассах и визуализируется в формате GIF.

### Скачивание результатов

Скачиваем ZIP-архив с сохранёнными весами нейросетей, графиками, GIF и результатами тестирования. Сам ноутбук `.ipynb` можно скачать отдельно через меню Jupyter или Google Colab.

In [ ]:
try:
    from google.colab import files
    files.download(str(output_dir) + '.zip')
except ImportError:
    print('Архив сохранён локально:', str(output_dir) + '.zip')